In [ ]:
import numpy as np
import pandas as pd
import transformers
import torch

from tqdm import tqdm, trange
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/CAN_Research/attack-free-1.csv', delimiter=',')
df['datetime'] = df['timestamp']
df['arbitration_id'] = df['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['data_field'] = df['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['attack'] = 0
df = df.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]


df.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,193.0,3.458765e+18,0
1,1.672531e+09,197.0,3.458765e+18,0
2,1.672531e+09,388.0,8.589935e+09,0
3,1.672531e+09,455.0,1.790802e+15,0
4,1.672531e+09,461.0,0.000000e+00,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50001 entries, 0 to 50000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   datetime        50001 non-null  float64
 1   arbitration_id  50001 non-null  float64
 2   data_field      50001 non-null  float64
 3   attack          50001 non-null  int64  
dtypes: float64(3), int64(1)
memory usage: 1.5 MB


In [ ]:
df.shape

(50001, 4)

In [ ]:
dg = pd.read_csv('/content/drive/MyDrive/CAN_Research/DoS-1.csv', delimiter=',')
dg['datetime'] = dg['timestamp']
dg['arbitration_id'] = dg['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['data_field'] = dg['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['attack'] = 1
dg = dg.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]

dg.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,485.0,5.044479e+18,1
1,1.672531e+09,489.0,4.503651e+15,1
2,1.672531e+09,249.0,1.225120e+17,1
3,1.672531e+09,761.0,6.313770e+11,1
4,1.672531e+09,409.0,1.498772e+19,1


In [ ]:
dg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50001 entries, 0 to 50000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   datetime        50001 non-null  float64
 1   arbitration_id  50001 non-null  float64
 2   data_field      50001 non-null  float64
 3   attack          50001 non-null  int64  
dtypes: float64(3), int64(1)
memory usage: 1.5 MB


In [ ]:
dg.shape

(50001, 4)

In [ ]:
data = pd.concat([df, dg])
data = shuffle(data)

In [ ]:
# Feature and label extraction
X = data[['datetime','arbitration_id', 'data_field']]
y = data['attack']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
len(data)

100002

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print("-" * 30)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Best Parameters:", model.best_params_)
    print("-" * 30)

In [ ]:
# Define parameter grids for each model
param_grids = {
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'saga'],
        'class_weight': ['balanced', None]
    },
    'DecisionTreeClassifier': {
        'max_depth': [None, 10, 20, 30, 40, 50],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', None]
    },
    'RandomForestClassifier': {
        'n_estimators': [10, 50, 100, 200],
        'max_features': ['auto', 'sqrt', 'log2'],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', None]
    },
    'SVC': {
        'C': [0.1, 1, 10, 100],
        'gamma': [1, 0.1, 0.01, 0.001],
        'kernel': ['rbf', 'linear'],
        'class_weight': ['balanced', None]
    },
    'KNeighborsClassifier': {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance']
    }
}

In [ ]:
# List of models to evaluate
models = [
    ('LogisticRegression', LogisticRegression()),
    ('DecisionTreeClassifier', DecisionTreeClassifier()),
    ('RandomForestClassifier', RandomForestClassifier()),
    ('SVC', SVC()),
    ('KNeighborsClassifier', KNeighborsClassifier())
]

# Evaluate each model with GridSearchCV
for name, model in models:
    grid_search = GridSearchCV(estimator=model, param_grid=param_grids[name],
                               scoring='f1', cv=5, n_jobs=-1)
    evaluate_model(grid_search, X_train, X_test, y_train, y_test)


------------------------------
Accuracy: 0.5107
Precision: 0.5080
Recall: 0.7406
F1 Score: 0.6027
Best Parameters: {'C': 0.01, 'class_weight': 'balanced', 'solver': 'saga'}
------------------------------
------------------------------
Accuracy: 0.7709
Precision: 0.7655
Recall: 0.7825
F1 Score: 0.7739
Best Parameters: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 10}
------------------------------


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(


------------------------------
Accuracy: 0.7414
Precision: 0.7545
Recall: 0.7173
F1 Score: 0.7354
Best Parameters: {'class_weight': 'balanced', 'max_depth': 10, 'max_features': 'auto', 'min_samples_split': 2, 'n_estimators': 10}
------------------------------
